# TCN Seizure Prediction
Cross-patient benchmarking on CHB-MIT · 20 seeds · mean ± std

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score
from data_utils import get_dataloaders_for_seed, SEEDS, DATA_DIR
from eval_utils import find_youden_threshold, full_evaluate

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 18
WIN          = 20 * 256
BATCH_SIZE   = 128
MAX_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4
OUT_DIR      = r'D:\seizure_results\tcn'

print(f'Device : {DEVICE}')
print(f'Seeds  : {len(SEEDS)}')
os.makedirs(OUT_DIR, exist_ok=True)

Device : cuda
Seeds  : 20


In [2]:
class TemporalBlock(nn.Module):
    """
    Single TCN residual block — two dilated causal Conv1d layers with
    weight norm, ReLU, and dropout, plus a residual connection.

    Weight-norm initialisation note:
    nn.utils.weight_norm decomposes weight into weight_v (direction) and
    weight_g (magnitude), recomputing weight = weight_g * weight_v/||weight_v||
    on every forward pass. Assigning to .weight after weight_norm is applied
    has no effect. We therefore initialise weight_v directly (small normal)
    and set weight_g to ones, matching the original locuslab/TCN intent.
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.utils.weight_norm(nn.Conv1d(
            in_ch, out_ch, kernel_size, dilation=dilation, padding=padding))
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(
            out_ch, out_ch, kernel_size, dilation=dilation, padding=padding))

        # Correct initialisation: target weight_v and weight_g, not weight
        nn.init.normal_(self.conv1.weight_v, 0, 0.01)
        nn.init.ones_(self.conv1.weight_g)
        nn.init.normal_(self.conv2.weight_v, 0, 0.01)
        nn.init.ones_(self.conv2.weight_g)

        self.net = nn.Sequential(
            self.conv1, nn.ReLU(), nn.Dropout(dropout),
            self.conv2, nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        out = out[:, :, :x.size(2)]   # causal trim
        return self.relu(out + self.residual(x))


class TCN(nn.Module):
    """
    Temporal Convolutional Network — Bai et al. (2018).
    Dilations: {1, 2, 4, 8}.

    Input  : (B, 18, 5120)
    Output : (B, 2)
    """
    def __init__(self,
                 n_channels  = N_CHANNELS,
                 n_filters   = 64,
                 kernel_size = 8,
                 dilations   = (1, 2, 4, 8),
                 dropout     = 0.2,
                 n_classes   = 2):
        super().__init__()
        layers = []
        in_ch  = n_channels
        for d in dilations:
            layers.append(TemporalBlock(in_ch, n_filters, kernel_size, d, dropout))
            in_ch = n_filters
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_filters, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.tcn(x)))


dummy = torch.zeros(4, N_CHANNELS, WIN)
model_test = TCN()
print('Output shape:', model_test(dummy).shape)   # expect (4, 2)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Output shape: torch.Size([4, 2])
Trainable parameters: 245,122


In [3]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


print('Helpers defined.')

Helpers defined.


In [4]:
def run_seed(seed):
    print(f"\n{'='*60}")
    print(f"  Seed {seed}")
    print(f"{'='*60}")

    (train_loader, val_loader, test_loader,
     n_train, n_val, n_test,
     n_pre, n_inter) = get_dataloaders_for_seed(seed, DATA_DIR, BATCH_SIZE)

    _bx, _by = next(iter(train_loader))
    _n1 = int(_by.sum())
    print(f"  Batch check : {_n1}/{len(_by)} pre-ictal ({100*_n1/len(_by):.0f}%)")
    del _bx, _by

    model     = TCN().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_val_auc  = 0.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_probs, val_labels = collect_probs(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_probs)
        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:3d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | lr {current_lr:.2e}")

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs, val_labels = collect_probs(model, val_loader)
    threshold = find_youden_threshold(val_labels, val_probs)
    print(f"  Youden threshold (val): {threshold:.4f}")

    test_probs, test_labels = collect_probs(model, test_loader)
    m = full_evaluate(test_labels, test_probs, threshold, stride_s=300)

    print(f"\n  [Seed {seed}] TEST  AUC {m['auc']:.4f} | "
          f"Sen {m['sensitivity']:.4f} | Spe {m['specificity']:.4f} | "
          f"Prec {m['precision']:.4f} | F1 {m['f1']:.4f} | "
          f"FAR {m['far']:.3f}/h | "
          f"EvtSen {m['event_sensitivity']:.3f} ({m['n_events']} events)")

    torch.save(best_state, os.path.join(OUT_DIR, f'seed{seed}_best.pt'))

    return {
        'seed':               seed,
        'test_auc':           m['auc'],
        'test_sen':           m['sensitivity'],
        'test_spe':           m['specificity'],
        'test_precision':     m['precision'],
        'test_f1':            m['f1'],
        'far':                m['far'],
        'event_sensitivity':  m['event_sensitivity'],
        'n_events':           m['n_events'],
        'threshold':          m['threshold'],
        'best_val_auc':       best_val_auc,
    }

print('run_seed defined.')

run_seed defined.


In [5]:
all_results = []
for s in SEEDS:
    result = run_seed(s)
    all_results.append(result)

results_path = os.path.join(OUT_DIR, 'results.json')
with open(results_path, 'w') as _f:
    json.dump(
        [{k: (int(v) if k in ('seed', 'n_events') else float(v))
          for k, v in r.items()}
         for r in all_results],
        _f, indent=2,
    )
print(f'\nAll results saved to {results_path}')


  Seed 42

[seed=42]  train=15pts  val=4pts  test=4pts
  train  total=11726  pre=5074  inter=6652  ratio=1:1
  val    total=3521  pre=2084  inter=1437
  test   total=2177  pre=1334  inter=843
  Batch check : 61/128 pre-ictal (48%)
  Epoch   1 | loss 0.7468 | val AUC 0.5199 | lr 1.00e-03
  Epoch   2 | loss 0.5524 | val AUC 0.4904 | lr 1.00e-03
  Epoch   3 | loss 0.5224 | val AUC 0.5027 | lr 1.00e-03
  Epoch   4 | loss 0.5272 | val AUC 0.5071 | lr 1.00e-03
  Epoch   5 | loss 0.5157 | val AUC 0.4952 | lr 1.00e-03
  Epoch   6 | loss 0.5501 | val AUC 0.5080 | lr 1.00e-03
  Epoch   7 | loss 0.4918 | val AUC 0.4725 | lr 5.00e-04
  Epoch   8 | loss 0.4549 | val AUC 0.4754 | lr 5.00e-04
  Epoch   9 | loss 0.4297 | val AUC 0.4804 | lr 5.00e-04
  Epoch  10 | loss 0.4174 | val AUC 0.4853 | lr 5.00e-04
  Epoch  11 | loss 0.4163 | val AUC 0.4863 | lr 5.00e-04
  Epoch  12 | loss 0.4133 | val AUC 0.4723 | lr 5.00e-04
  Epoch  13 | loss 0.4634 | val AUC 0.4769 | lr 2.50e-04
  Epoch  14 | loss 0.3876 |

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch   1 | loss 0.7134 | val AUC 0.5754 | lr 1.00e-03
  Epoch   2 | loss 0.5824 | val AUC 0.5317 | lr 1.00e-03
  Epoch   3 | loss 0.5727 | val AUC 0.5947 | lr 1.00e-03
  Epoch   4 | loss 0.5433 | val AUC 0.6016 | lr 1.00e-03
  Epoch   5 | loss 0.5214 | val AUC 0.5862 | lr 1.00e-03
  Epoch   6 | loss 0.5093 | val AUC 0.6132 | lr 1.00e-03
  Epoch   7 | loss 0.5114 | val AUC 0.5543 | lr 1.00e-03
  Epoch   8 | loss 0.5149 | val AUC 0.6030 | lr 1.00e-03
  Epoch   9 | loss 0.5272 | val AUC 0.6019 | lr 1.00e-03
  Epoch  10 | loss 0.5106 | val AUC 0.6115 | lr 1.00e-03
  Epoch  11 | loss 0.4810 | val AUC 0.5809 | lr 1.00e-03
  Epoch  12 | loss 0.4684 | val AUC 0.5778 | lr 5.00e-04
  Epoch  13 | loss 0.4542 | val AUC 0.5571 | lr 5.00e-04
  Epoch  14 | loss 0.4248 | val AUC 0.5703 | lr 5.00e-04
  Epoch  15 | loss 0.4258 | val AUC 0.5381 | lr 5.00e-04
  Epoch  16 | loss 0.4142 | val AUC 0.5330 | lr 5.00e-04
  Epoch  17 | loss 0.4510 | val AUC 0.5413 | lr 5.00e-04
  Epoch  18 | loss 0.4303 | val

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch   1 | loss 0.7083 | val AUC 0.4914 | lr 1.00e-03
  Epoch   2 | loss 0.6482 | val AUC 0.4976 | lr 1.00e-03
  Epoch   3 | loss 0.5705 | val AUC 0.4798 | lr 1.00e-03
  Epoch   4 | loss 0.5612 | val AUC 0.5043 | lr 1.00e-03
  Epoch   5 | loss 0.5568 | val AUC 0.5055 | lr 1.00e-03
  Epoch   6 | loss 0.5225 | val AUC 0.5287 | lr 1.00e-03
  Epoch   7 | loss 0.5484 | val AUC 0.5410 | lr 1.00e-03
  Epoch   8 | loss 0.5024 | val AUC 0.5580 | lr 1.00e-03
  Epoch   9 | loss 0.4826 | val AUC 0.5240 | lr 1.00e-03
  Epoch  10 | loss 0.5156 | val AUC 0.5570 | lr 1.00e-03
  Epoch  11 | loss 0.4834 | val AUC 0.5437 | lr 1.00e-03
  Epoch  12 | loss 0.4748 | val AUC 0.5608 | lr 1.00e-03
  Epoch  13 | loss 0.4728 | val AUC 0.5453 | lr 1.00e-03
  Epoch  14 | loss 0.4564 | val AUC 0.5450 | lr 1.00e-03
  Epoch  15 | loss 0.5069 | val AUC 0.5457 | lr 1.00e-03
  Epoch  16 | loss 0.4741 | val AUC 0.5509 | lr 1.00e-03
  Epoch  17 | loss 0.4641 | val AUC 0.5496 | lr 1.00e-03
  Epoch  18 | loss 0.4542 | val

...

  Epoch   1 | loss 0.6966 | val AUC 0.5320 | lr 1.00e-03
  Epoch   2 | loss 0.5805 | val AUC 0.6020 | lr 1.00e-03
  Epoch   3 | loss 0.5733 | val AUC 0.5846 | lr 1.00e-03
  Epoch   4 | loss 0.5340 | val AUC 0.6039 | lr 1.00e-03
  Epoch   5 | loss 0.5446 | val AUC 0.4893 | lr 1.00e-03
  Epoch   6 | loss 0.5109 | val AUC 0.5926 | lr 1.00e-03
  Epoch   7 | loss 0.5157 | val AUC 0.5853 | lr 1.00e-03
  Epoch   8 | loss 0.4604 | val AUC 0.5371 | lr 1.00e-03
  Epoch   9 | loss 0.4667 | val AUC 0.5415 | lr 1.00e-03
  Epoch  10 | loss 0.4675 | val AUC 0.5409 | lr 5.00e-04
  Epoch  11 | loss 0.4357 | val AUC 0.5161 | lr 5.00e-04
  Epoch  12 | loss 0.4376 | val AUC 0.5628 | lr 5.00e-04
  Epoch  13 | loss 0.4356 | val AUC 0.5330 | lr 5.00e-04
  Epoch  14 | loss 0.4167 | val AUC 0.5441 | lr 5.00e-04
  Epoch  15 | loss 0.4338 | val AUC 0.5585 | lr 5.00e-04
  Epoch  16 | loss 0.4104 | val AUC 0.5281 | lr 2.50e-04
  Epoch  17 | loss 0.3845 | val AUC 0.4999 | lr 2.50e-04
  Epoch  18 | loss 0.3891 | val

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch   1 | loss 0.6837 | val AUC 0.5054 | lr 1.00e-03
  Epoch   2 | loss 0.6114 | val AUC 0.5209 | lr 1.00e-03
  Epoch   3 | loss 0.5476 | val AUC 0.4989 | lr 1.00e-03
  Epoch   4 | loss 0.5430 | val AUC 0.4775 | lr 1.00e-03
  Epoch   5 | loss 0.5451 | val AUC 0.4972 | lr 1.00e-03
  Epoch   6 | loss 0.5182 | val AUC 0.4975 | lr 1.00e-03
  Epoch   7 | loss 0.5105 | val AUC 0.4924 | lr 1.00e-03
  Epoch   8 | loss 0.5063 | val AUC 0.5001 | lr 5.00e-04
  Epoch   9 | loss 0.4846 | val AUC 0.5017 | lr 5.00e-04
  Epoch  10 | loss 0.4640 | val AUC 0.4919 | lr 5.00e-04
  Epoch  11 | loss 0.4410 | val AUC 0.4854 | lr 5.00e-04
  Epoch  12 | loss 0.4305 | val AUC 0.4770 | lr 5.00e-04
  Epoch  13 | loss 0.4392 | val AUC 0.4888 | lr 5.00e-04
  Epoch  14 | loss 0.4364 | val AUC 0.4675 | lr 2.50e-04
  Epoch  15 | loss 0.4134 | val AUC 0.4886 | lr 2.50e-04
  Epoch  16 | loss 0.4130 | val AUC 0.4805 | lr 2.50e-04
  Epoch  17 | loss 0.4003 | val AUC 0.4851 | lr 2.50e-04
  Epoch  18 | loss 0.3908 | val

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch   1 | loss 0.7173 | val AUC 0.4798 | lr 1.00e-03
  Epoch   2 | loss 0.6276 | val AUC 0.4599 | lr 1.00e-03
  Epoch   3 | loss 0.5844 | val AUC 0.4833 | lr 1.00e-03
  Epoch   4 | loss 0.5571 | val AUC 0.4905 | lr 1.00e-03
  Epoch   5 | loss 0.5425 | val AUC 0.5184 | lr 1.00e-03
  Epoch   6 | loss 0.5582 | val AUC 0.5191 | lr 1.00e-03
  Epoch   7 | loss 0.5579 | val AUC 0.4775 | lr 1.00e-03
  Epoch   8 | loss 0.5363 | val AUC 0.4865 | lr 1.00e-03
  Epoch   9 | loss 0.5228 | val AUC 0.5577 | lr 1.00e-03
  Epoch  10 | loss 0.5180 | val AUC 0.5438 | lr 1.00e-03
  Epoch  11 | loss 0.5221 | val AUC 0.5349 | lr 1.00e-03
  Epoch  12 | loss 0.5080 | val AUC 0.5548 | lr 1.00e-03
  Epoch  13 | loss 0.4994 | val AUC 0.5535 | lr 1.00e-03
  Epoch  14 | loss 0.5277 | val AUC 0.5548 | lr 1.00e-03
  Epoch  15 | loss 0.5212 | val AUC 0.5551 | lr 5.00e-04
  Epoch  16 | loss 0.4749 | val AUC 0.5200 | lr 5.00e-04
  Epoch  17 | loss 0.4738 | val AUC 0.5481 | lr 5.00e-04
  Epoch  18 | loss 0.4786 | val

In [6]:
metrics = ['test_auc', 'test_sen', 'test_spe', 'test_precision',
           'test_f1', 'far', 'event_sensitivity']
labels  = ['AUC', 'Sensitivity (win)', 'Specificity', 'Precision',
           'F1', 'FAR (/h)', 'Sensitivity (event)']

print(f'TCN  Cross-Patient Results (mean ± std, n={len(all_results)} seeds)')
print('-' * 55)
for m, l in zip(metrics, labels):
    vals = np.array([r[m] for r in all_results])
    print(f'  {l:<22s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}')

TCN  Cross-Patient Results (mean ± std, n=20 seeds)
-------------------------------------------------------
  AUC                   : 0.5735 ± 0.0586
  Sensitivity (win)     : 0.4836 ± 0.2264
  Specificity           : 0.6087 ± 0.2147
  Precision             : 0.5896 ± 0.1164
  F1                    : 0.4966 ± 0.1578
  FAR (/h)              : 4.6962 ± 2.5767
  Sensitivity (event)   : 0.9107 ± 0.1065
